# Q6: Modeling Preparation

**Phase 7:** Modeling Preparation  
**Points: 3 points**

**Focus:** Perform temporal train/test split, select features, handle
categorical variables.

**Lecture Reference:** See **Lecture 11, Notebook 3**
(`11/demo/03_pattern_analysis_modeling_prep.ipynb`), Phase 7 for
examples of temporal train/test splitting and feature preparation.
**CRITICAL:** The lecture emphasizes why temporal splitting is required
(not random split) for time series data.

## Setup

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Load feature-engineered data from Q4
df = pd.read_csv('output/q4_features.csv', parse_dates=['Measurement Timestamp'], index_col='Measurement Timestamp')

# Or if you saved without index:
# df = pd.read_csv('output/q4_features.csv')
# df['Measurement Timestamp'] = pd.to_datetime(df['Measurement Timestamp'])
# df = df.set_index('Measurement Timestamp')
print(f"Loaded {len(df):,} records with features")

In [ ]:
# Select target variable
target = "Air Temperature"

# List all columns
all_columns = df.columns.tolist()

# Remove target from potential features or leakage/ID cols
features_to_exclude = [
    "Air Temperature",
    "Air Temperature (F)",
    "Wet Bulb Temperature",
    "Comfort Index",
    "Temp Ratio",
    "Temperature Difference",
    "Air Temperature Categories",
    "Measurement ID",
    "Measurement Timestamp Label",
]

# Add any target-derived features to exclude (adjust based on your features)
for col in all_columns:
    if col != target:
        # Check if feature is derived from target (high correlation check)
        if target.lower().replace(' ', '_') in col.lower():
            features_to_exclude.append(col)
            print(f"Excluding potentially circular feature: {col}")

# Get feature columns
feature_columns = [col for col in all_columns if col not in features_to_exclude]
print(f"\nNumber of features: {len(feature_columns)}")
print(f"Features: {feature_columns}")

# Prepare X (features) and y (target)
X = df[feature_columns].copy()
y = df[target].copy()

# Check for missing values
print(f"\nMissing values in X: {X.isnull().sum().sum()}")
print(f"Missing values in y: {y.isnull().sum()}")
# Do not drop missing rows of q7_modeling will not work

# One-hot encode categorical variables
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
if len(categorical_columns) > 0:
    X = pd.get_dummies(X, columns=categorical_columns, drop_first=True)
print(f"After encoding, number of features: {X.shape[1]}")

In [ ]:
# Perform temporal train/test split
# CRITICAL: Sort by datetime index to ensure temporal order
X = X.sort_index()
y = y.sort_index()

# Calculate split point (80/20 split)
split_ratio = 0.8
split_index = int(len(X) * split_ratio)

# Split into train and test (temporal split)
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# Get date ranges for documentation
train_start = X_train.index.min()
train_end = X_train.index.max()
test_start = X_test.index.min()
test_end = X_test.index.max()

# Print split information
print(f"\n{'='*50}")
print("TRAIN/TEST Split Method")
print(f"{'='*50}")
print(f"Split Method: Temporal (80/20 split by time)")
print(f"\nTraining Set Size: {len(X_train):,} samples")
print(f"Test Set Size: {len(X_test):,} samples")
print(f"\nTraining Date Range: {train_start} to {train_end}")
print(f"Test Date Range: {test_start} to {test_end}")
print(f"\nNumber of Features: {X_train.shape[1]}")
print(f"Target Variable: {target}")


In [ ]:
# Save artifacts

# Save X_train (no index, no datetime)
X_train.to_csv('output/q6_X_train.csv', index=False)
print(f"\nSaved: output/q6_X_train.csv")

# Save X_test (no index, no datetime)
X_test.to_csv('output/q6_X_test.csv', index=False)
print(f"Saved: output/q6_X_test.csv")

# Save y_train (single column with target name as header, no index)
y_train.to_csv('output/q6_y_train.csv', index=False, header=True)
print(f"Saved: output/q6_y_train.csv")

# Save y_test (single column with target name as header, no index)
y_test.to_csv('output/q6_y_test.csv', index=False, header=True)
print(f"Saved: output/q6_y_test.csv")

# Save train/test split information
info_text = f"""TRAIN/TEST SPLIT INFORMATION
==========================

Split Method: Temporal (80/20 split by time)

Training Set Size: {len(X_train)} samples
Test Set Size: {len(X_test)} samples

Training Date Range: {train_start} to {train_end}
Test Date Range: {test_start} to {test_end}

Number of Features: {X_train.shape[1]}
Target Variable: {target}

Feature List:
{chr(10).join([f'- {col}' for col in X_train.columns])}
"""

with open('output/q6_train_test_info.txt', 'w') as f:
    f.write(info_text)
print(f"Saved: output/q6_train_test_info.txt")

# Step 7: Verification
print(f"\n{'='*50}")
print("VERIFICATION")
print(f"{'='*50}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"\nNo data leakage: Training data ends before test data begins")
print(f"Temporal order preserved: {train_end < test_start}")

# Check for any overlap
print(f"\nNo temporal overlap between train and test: {train_end < test_start}")

print(f"\n{'='*50}")
print("Q6 MODELING PREPARATION COMPLETE")
print(f"{'='*50}")
print("\nAll 5 required artifacts saved:")
print("✓ output/q6_X_train.csv")
print("✓ output/q6_X_test.csv")
print("✓ output/q6_y_train.csv")
print("✓ output/q6_y_test.csv")
print("✓ output/q6_train_test_info.txt")